大模型具有不确定性，尤其是构建复杂`AI Agent`应用程序中，中间会涉及非常多的子步骤，开发者可能会想要了解每一步的运行状态和结果
`Langsmith`是由​​ `LangChain` 和 `LangGraph` 背后的团队创建，**主要作用是：为基于大语言模型构建的应用程序提供全面的监控、调试和可观察性。提供强大的跟踪、日志记录和实时分析功能。

对于一个项目而言，可以是单个应用程序或服务。该项目将包含多个跟踪，每个跟踪都是运行的集合 
* 一个运行代表应用程序中的一个基本操作，例如对 OpenAI 的 API 调用，或检索操作。如下图所示：

<div align=center><img src="./pic/lesson03/1.png" width=100%></div>

- **Project (项目)** 蓝色方块代表整个项目，可能是一个单独的应用程序或服务。
- **Traces (轨迹)** 绿色方块代表项目在不同条件或配置下的执行路径。每个轨迹可以是一个特定的用户会话、一个功能的执行，或者应用在特定输入下的行为。
- **Runs (运行)** 每个轨迹下的黄色方块表示特定轨迹的单次执行。这些是执行的实例，每个实例都是轨迹在特定条件下的实际运行。
- **Feedback, Tags, Metadata (反馈、标签、元数据)** 这部分显示了系统如何利用用户或自动化工具生成的反馈、标签和元数据来增强轨迹的管理和过滤。反馈可以用于改进未来的运行，标签和元数据可用于分类和筛选特定的轨迹或运行，以便在LangSmith的用户界面中更容易地管理和审查

```python
"""
这里是示例, 仅仅为了解释, 实际见.env 文件
"""
os.environ["LANGCHAIN_TRACING_V2"] = "true" # 是否启用
os.environ["LANGCHAIN_API_KEY"] = "lsv2***2" # api_key, 官网配置
os.environ["LANGCHAIN_PROJECT"] = "chat" # 项目名称, 会自动创建, 若不配置, 自动存入 default 项目
"""
追踪数据发送的服务器地址, 这里是 LangSmith 官方公有云服务器, 不需要配置, 会自动适用
若需要使用其它服务器或私有服务器, 则需要配置响应的地址
"""
# os.environ["LANGCHAIN_ENDPOINT"]= "https://api.smith.langchain.com"
```

In [2]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI

import os


qwen_llm = ChatOpenAI(
    model="qwen3.7-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


def chat_bot(state: MessagesState):
    return {"messages": [qwen_llm.invoke(state["messages"])]}


builder = StateGraph(MessagesState)

builder.add_node("chat_bot", chat_bot)

builder.add_edge(START, "chat_bot")
builder.add_edge("chat_bot", END)

graph = builder.compile()


def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [("user", user_input)]}):
        for value in event.values():
            print(f"Qwen: {value['messages'][-1].content}")


while True:
    try:
        user_input = input("用户提问: ")
        if user_input.lower() in ["退出", "quit", "q", "再见"]:
            break
        print(f"User: {user_input}")
        stream_graph_updates(user_input=user_input)
    except Exception as e:
        print("Error: {e}")

User: 你好
Qwen: 你好！有什么我可以帮你的吗？
User: 谈一下如何运营自己的 X 账号
Qwen: 运营 X（原 Twitter）账号是一项系统工程，与微信公众号的“深度阅读”或小红书的“视觉种草”不同，**X 的核心逻辑是“观点碰撞”、“信息密度”和“真实人设”**。

以下是一套从 0 到 1 再到变现的 X 账号运营全指南：

---

### 第一阶段：想清楚再动手（定位与基建）

#### 1. 明确账号定位（你是谁？提供什么价值？）
在 X 上，泛泛而谈的账号很难出头，你必须有一个**垂直且清晰的标签**。
*   **细分领域**：AI 应用、独立开发、Web3、个人成长、出海营销、特定行业观察等。
*   **人设打造**：X 用户极度偏爱 **“真实的人”** 和 **“Build in public（公开构建）”**。不要做冷冰冰的机构号，要展示你的思考、失败、日常和性格。
*   **价值主张**：关注你能给粉丝带来什么？是“提供干货”、“提供情绪价值”还是“提供前沿资讯”？

#### 2. 账号基建（装修你的门面）
*   **头像**：最好是**清晰的真人照片**（带点微笑或专业感），或者极具辨识度的 IP 形象。真人头像信任度最高。
*   **名字与 Handle**：名字好记，Handle（@后面的ID）尽量简短，不要用一堆乱码数字。
*   **Bio（简介）**：这是你的“电梯演讲”。公式：**我是谁 + 我做过什么（背书） + 我能为你提供什么价值 + CTA（行动呼吁/链接）**。
    *   *示例*：“连续创业者 | 专注 AI 落地应用 | 分享独立开发日常与搞钱思考 | 订阅我的周刊 👇”
*   **置顶推文（Pin）**：放你**数据最好的一篇 Thread（长推文）**、你的**个人高光故事**、或者一个**高价值的资源合集**。这是新访客转化粉丝的关键。

---

### 第二阶段：内容策略（核心驱动力）

#### 1. 内容形式与比例
*   **Thread（长推文/串推）**：**涨粉利器**。适合深度拆解、教程、复盘。每周至少 1-2 篇高质量 Thread。
*   **短推文（Short Tweets）**：保持日常活跃度。金句、观点、行业短评、日常随想。每天 1-3 条。

业务代码完全不需要做任何改动，不需要写装饰器，也不需要写打点上报的代码
这种设计在软件架构中被称为 “零代码侵入（Zero-code Instrumentation）”

在 LangChain 和 LangGraph 中，几乎所有核心对象——无论是 ChatOpenAI、PromptTemplate，还是 StateGraph 的节点和工具——在底层都继承自同一个基类：Runnable。

每当你调用 .invoke() 或 .stream() 时，基类内部其实暗地里做了包装，伪代码类似于这样：

```python
def invoke(self, input):
    # 触发钩子：告诉监听器“我开始跑了，这是输入”
    callback_manager.on_chain_start(input)
    
    try:
        # 执行你真正的业务逻辑（比如调用大模型）
        output = self._call(input)
        
        # 触发钩子：告诉监听器“我跑完了，这是输出和耗时”
        callback_manager.on_chain_end(output)
        return output
    except Exception as e:
        # 触发钩子：如果报错了，记录报错信息
        callback_manager.on_chain_error(e)
        raise e
```

也就是说：每一个组件的身上，出生时就已经被装好了“全套探针”


LangChain 在被 Python 导入加载的一瞬间，就会去扫描系统环境变量 `LANGCHAIN_TRACING_V2`。一旦发现这个变量是 "true"，它就会立刻在后台全局激活一个叫 LangChainTracer 的监听器，把之前埋好的所有探针全部“通电唤醒”！


LangSmith 采用了后台异步队列机制：
- 探针收集到数据后，只是极快地把它塞进本地内存队列
- 后台有一个专门的轻量级守护线程，负责把数据打包并批量异步上传
- 即使你的网络极其卡顿甚至断网，你的主程序也会丝滑运行，绝对不会因为追踪失败而导致业务报错